In [1]:
#Nếu chạy trên Google Colab thì cần kết nối với máy chủ trước
from google.colab import drive #type :ignore
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

In [3]:
import tensorflow as tf
from tensorflow import keras
print(tf.__version__)

2.19.0


In [14]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, num_channels, use_1x1conv=False, strides=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, num_channels, kernel_size=3, padding = 1,stride = strides)
        self.conv2 = nn.Conv2d(num_channels,num_channels, kernel_size=3, padding = 1)
        self.conv3 = None # Đừng sửa None này nhé :!
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels,num_channels,kernel_size = 1,stride = strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = nn.ReLU()(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        return nn.ReLU()(Y + X)

In [5]:
X = torch.randn((4, 3, 6, 6)) 
blk = Residual(in_channels = 3, num_channels = 3)
assert blk(X).shape  == (4,3,6,6)

In [6]:
net = nn.Sequential()
net.add_module("conv",nn.Conv2d(1,64, kernel_size=7, stride=2, padding=3))
net.add_module("batchnorm",nn.BatchNorm2d(64))
net.add_module("Relu",nn.ReLU())
net.add_module("maxpool",nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1))

In [16]:
def resnet_block(in_channels ,num_channels, num_residuals, first_block=False):
    blk = nn.Sequential()
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.add_module('residual_{}'.format(i),Residual(in_channels , num_channels, use_1x1conv=True, strides = 2))
        else:
            current_in = in_channels if i == 0 else num_channels
            blk.add_module('residual_{}'.format(i),Residual(current_in, num_channels))
    return blk

In [17]:
net.add_module('resnet_block1',resnet_block(64, 64, 2, first_block=True))
net.add_module('resnet_block2',resnet_block(64,128, 2))
net.add_module('resnet_block3',resnet_block(128,256, 2))
net.add_module('resnet_block4',resnet_block(256,512, 2))

In [18]:
net.add_module('GlobalAverage', nn.AdaptiveAvgPool2d((1,1)))
net.add_module('Flatten', nn.Flatten())
net.add_module('Linear', nn.Linear(512,10))

In [19]:
X = torch.randn((1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t', X.shape)

Conv2d output shape:	 torch.Size([1, 64, 112, 112])
BatchNorm2d output shape:	 torch.Size([1, 64, 112, 112])
ReLU output shape:	 torch.Size([1, 64, 112, 112])
MaxPool2d output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 128, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 512, 1, 1])
Flatten output shape:	 torch.Size([1, 512])
Linear output shape:	 torch.Size([1, 10])


In [20]:
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import random

In [22]:
epochs = 20

# Các tham số cần thiết cho quá trình traning.
learning_rate = 0.01
batch_size = 160
display_step = 10

# Path lưu best model 
checkpoint = 'model.pth' # có thể để dạng *.pth

# device chúng ta dùng cuda
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda' 

In [23]:
# Transform image 
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
    ])

# load dataset từ torchvision.datasets
train_dataset = datasets.MNIST('../data', train=True, download=True,transform=transform)
test_dataset = datasets.MNIST('../data', train=False,transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size)

In [24]:
# call model, set deivce
model = net.to(device)
# load lại pretrained model (nếu có)
try:
  None
except:
  print("!!! Hãy train để có checkpoint file")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters() , lr = learning_rate)
best_val_loss = 999

for epoch in range(1,epochs):
    # Quá trình training 
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device),target.to(device) # device?
        optimizer.zero_grad() # Zero_grad
        output = model(data)
        loss = criterion(output, target)
        loss.backward() # backward
        optimizer.step() # update weights
        if batch_idx % display_step == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tTrain Loss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
    # Quá trình testing 
    model.eval()
    test_loss = 0
    correct = 0
    # set no grad cho quá trình testing
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            output = F.log_softmax(output, dim = 1) # log softmax using F, chu y dim nhe
            test_loss += criterion(output, target).item()
            pred =  output.argmax(dim = 1, keepdim = True)# argmax để lấy predicted label, chú ý keepdim = True
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset) 
    if test_loss < best_val_loss:
      best_val_loss = test_loss 
      torch.save(model.state_dict(), 'model.pth') # lưu lại model 
      print("***********    TEST_ACC = {}%    ***********".format(correct))

Train Epoch: 1 [0/60000 (0%)]	Train Loss: 2.299356
Train Epoch: 1 [1600/60000 (3%)]	Train Loss: 1.399824
Train Epoch: 1 [3200/60000 (5%)]	Train Loss: 0.783292
Train Epoch: 1 [4800/60000 (8%)]	Train Loss: 0.665521
Train Epoch: 1 [6400/60000 (11%)]	Train Loss: 0.423303
Train Epoch: 1 [8000/60000 (13%)]	Train Loss: 0.280115
Train Epoch: 1 [9600/60000 (16%)]	Train Loss: 0.147357
Train Epoch: 1 [11200/60000 (19%)]	Train Loss: 0.239762
Train Epoch: 1 [12800/60000 (21%)]	Train Loss: 0.260670
Train Epoch: 1 [14400/60000 (24%)]	Train Loss: 0.159672
Train Epoch: 1 [16000/60000 (27%)]	Train Loss: 0.194551
Train Epoch: 1 [17600/60000 (29%)]	Train Loss: 0.173725
Train Epoch: 1 [19200/60000 (32%)]	Train Loss: 0.125504
Train Epoch: 1 [20800/60000 (35%)]	Train Loss: 0.101056
Train Epoch: 1 [22400/60000 (37%)]	Train Loss: 0.172450
Train Epoch: 1 [24000/60000 (40%)]	Train Loss: 0.085996
Train Epoch: 1 [25600/60000 (43%)]	Train Loss: 0.180332
Train Epoch: 1 [27200/60000 (45%)]	Train Loss: 0.109789
Train 

AttributeError: 'int' object has no attribute 'to'